# Router full-all: picks vs what each future slot actually decodes to

**Model:** router full-all (`runs/libero_joint_2cam224_1e-4_router_full_all/2026-10-12_22-29-10`, step 21700).
- **Training:** video loss on all three futures; the action reads the one future chosen by the learned router; no modality prompt.

**Per window** (every `STRIDE` frames along an episode, plus a cross-task sample):
1. **Router pass:** `infer_joint` with no `active_future`, so the router picks as in eval. Its probabilities are recorded from the model's own `_sample_modality_select` call, and the picked future is decoded.
2. **Forced passes:** `infer_joint` with `active_future` set to each of the other two streams, which decodes those slots too (same seed).
3. **Content check:** each decoded future (frames 1..T) is compared with the ground-truth RGB, depth and flow futures (PSNR); the best match is what the slot **actually contains**.

**Question:** does the router pick between genuinely different futures, and what does the slot it picks really look like?

The model's computation is unchanged: the router call is wrapped from this notebook only to record its output. No repo file is modified. Set `QUICK = True` (or env `RPICK_QUICK=1`) for a smoke run.

In [ ]:
import os, time, collections
import numpy as np
import torch
import matplotlib.pyplot as plt

REPO = "/data/mgovind/FastWAM"
os.chdir(REPO)
os.environ.setdefault("DIFFSYNTH_MODEL_BASE_PATH", f"{REPO}/checkpoints")
os.environ.setdefault("DIFFSYNTH_DOWNLOAD_SOURCE", "huggingface")

TASK = "libero_joint_2cam224_1e-4_router_full_all"
RUN = "runs/libero_joint_2cam224_1e-4_router_full_all/2026-10-12_22-29-10"
STEP = 21700
EPISODE = 0          # episode for the trajectory view (0 = first spatial episode)
STRIDE = 10          # frames between windows (= eval replan)
N_CROSS = 40         # cross-task sample: first window of N_CROSS episodes spread over the dataset
STEPS = 10           # denoising steps (= eval)
SEED = 42
LAYOUT = "short"     # inference layout (short == full for this model, verified)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
QUICK = os.environ.get("RPICK_QUICK", "0") == "1"
if QUICK:
    STRIDE, N_CROSS, STEPS = 50, 4, 2
OUT = "notebooks/router_analysis/outputs/router_picks_decoding"
os.makedirs(OUT, exist_ok=True)
STREAMS = ["rgb", "depth", "flow"]
print(DEVICE, DTYPE, "QUICK" if QUICK else "")

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from fastwam.utils.video_metrics import pil_frames_to_video_tensor, video_psnr

with initialize_config_dir(version_base="1.3", config_dir=f"{REPO}/configs"):
    cfg = compose(config_name="train", overrides=[f"task={TASK}"])
t0 = time.time()
model = instantiate(cfg.model, model_dtype=DTYPE, device=DEVICE)
model.load_checkpoint(f"{RUN}/checkpoints/weights/step_{STEP:06d}.pt")
model = model.to(DEVICE).eval()
model.router_layout = LAYOUT
ds = instantiate(cfg.data.train, pretrained_norm_stats=f"{RUN}/dataset_stats.json")
if hasattr(ds, "skip_padding_as_possible"):
    ds.skip_padding_as_possible = False
edi = ds.lerobot_dataset.episode_data_index
ep_from, ep_to = edi["from"].tolist(), edi["to"].tolist()
print(f"{type(model).__name__} ({model.modality_select}, layout={model.router_layout}) loaded in {time.time()-t0:.0f}s; "
      f"{len(ep_from)} episodes")

In [ ]:
_orig_select = model._sample_modality_select
_rec = {}
def _recording_select(*a, **k):
    out = _orig_select(*a, **k)
    choices, logits, _ = out
    if logits is not None:
        _rec["probs"] = torch.softmax(logits.float(), -1)[0].cpu().numpy()
        _rec["pick"] = int(choices[0])
    return out

def to01(v):
    return ((v.float().clamp(-1, 1) + 1) / 2)

def decode(sample, active_future):
    kw = dict(prompt=None, input_image=sample["video_rgb"][:, 0].to(DEVICE, DTYPE),
              num_video_frames=int(sample["video_rgb"].shape[1]), action_horizon=int(sample["action"].shape[0]),
              proprio=sample["proprio"][0].to(DEVICE), context=sample["context"].to(DEVICE),
              context_mask=sample["context_mask"].to(DEVICE), num_inference_steps=STEPS, seed=SEED)
    if active_future is not None:
        kw["active_future"] = active_future
    with torch.no_grad():
        out = model.infer_joint(**kw)
    return out["active_future"], pil_frames_to_video_tensor(out["video"])   # [3, T, H, W] in [0, 1]

def analyse(sample):
    """Router pick + probabilities, decoded future of every slot, and which GT modality each slot matches."""
    model._sample_modality_select = _recording_select
    try:
        _rec.clear()
        picked, vid = decode(sample, None)                 # router decides, like eval
    finally:
        model._sample_modality_select = _orig_select
    probs, pick = _rec["probs"], STREAMS[_rec["pick"]]
    assert picked == pick, (picked, pick)
    vids = {pick: vid}
    for st in STREAMS:
        if st != pick:
            vids[st] = decode(sample, st)[1]
    gts = {g: to01(sample[f"video_{g}"]) for g in STREAMS}
    psnr = {st: {g: float(video_psnr(pred=vids[st][:, 1:], target=gts[g][:, 1:])) for g in STREAMS} for st in STREAMS}
    looks = {st: max(psnr[st], key=psnr[st].get) for st in STREAMS}
    return dict(pick=pick, probs=probs, vids=vids, psnr=psnr, looks=looks, sample=sample)
print("ok")

In [ ]:
# Trajectory view: one window every STRIDE frames over EPISODE
a, b = ep_from[EPISODE], ep_to[EPISODE]
frames = list(range(0, b - a, STRIDE))
traj = []
t0 = time.time()
for i, t in enumerate(frames):
    r = analyse(ds[a + t]); r["t"] = t; traj.append(r)
    print(f"frame {t:4d}: router picks {r['pick']:5s} p={np.round(r['probs'], 2)} | slots look like "
          + ", ".join(f"{s}->{r['looks'][s]}" for s in STREAMS) + f"  ({time.time()-t0:.0f}s)")
np.savez(f"{OUT}/ep{EPISODE}_traj.npz", frames=np.array(frames), probs=np.stack([r['probs'] for r in traj]),
         pick=np.array([STREAMS.index(r['pick']) for r in traj]),
         looks=np.array([[STREAMS.index(r['looks'][s]) for s in STREAMS] for r in traj]),
         psnr=np.array([[[r['psnr'][s][g] for g in STREAMS] for s in STREAMS] for r in traj]))

In [ ]:
# Plot A: router probabilities over the episode + what the picked slot contains
COL = {"rgb": "tab:red", "depth": "tab:blue", "flow": "tab:green"}
P = np.stack([r["probs"] for r in traj]); T = [r["t"] for r in traj]
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True, gridspec_kw=dict(height_ratios=[3, 1]))
axes[0].stackplot(T, P.T, labels=[f"p({s})" for s in STREAMS], colors=[COL[s] for s in STREAMS], alpha=0.6)
axes[0].set_ylabel("router probability"); axes[0].legend(loc="upper right", fontsize=8); axes[0].set_ylim(0, 1)
axes[0].set_title(f"Router full-all, episode {EPISODE}: router choice per replan")
for r in traj:
    axes[1].scatter(r["t"], 1, color=COL[r["pick"]], s=80, marker="s")
    axes[1].scatter(r["t"], 0, color=COL[r["looks"][r["pick"]]], s=80, marker="o")
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(["picked slot\nlooks like", "router pick"]); axes[1].set_xlabel("episode frame")
plt.tight_layout(); plt.savefig(f"{OUT}/ep{EPISODE}_router_timeline.png", dpi=120); plt.show()

In [ ]:
# Plot B: per window, the three decoded futures (last frame), picked slot boxed, GT for reference
def show_window(r):
    fr = r["vids"]["rgb"].shape[1] - 1
    fig, axes = plt.subplots(2, 3, figsize=(12, 4.6))
    for j, st in enumerate(STREAMS):
        ax = axes[0, j]; ax.imshow(r["vids"][st][:, fr].permute(1, 2, 0).numpy()); ax.axis("off")
        tag = "PICKED  " if st == r["pick"] else ""
        ax.set_title(f"{tag}{st} slot (p={r['probs'][j]:.2f})\nlooks like {r['looks'][st]} "
                     f"({r['psnr'][st][r['looks'][st]]:.1f} dB)", fontsize=8, color="crimson" if tag else "k")
        if tag:
            for s in ax.spines.values(): s.set_visible(True); s.set_color("crimson"); s.set_linewidth(3)
            ax.axis("on"); ax.set_xticks([]); ax.set_yticks([])
        g = to01(r["sample"][f"video_{st}"])[:, fr].permute(1, 2, 0).numpy()
        axes[1, j].imshow(g); axes[1, j].set_title(f"GT {st}", fontsize=8); axes[1, j].axis("off")
    fig.suptitle(f"episode {EPISODE}, frame {r['t']}: router picks {r['pick']}", fontsize=10)
    plt.tight_layout(); plt.savefig(f"{OUT}/ep{EPISODE}_frame{r['t']:04d}.png", dpi=110); plt.show(); plt.close("all")

for r in traj:
    show_window(r)

In [ ]:
# Cross-task sample: first window of N_CROSS episodes spread over the dataset
eps = np.linspace(0, len(ep_from) - 1, N_CROSS).round().astype(int).tolist()
cross = []
t0 = time.time()
for i, e in enumerate(eps):
    s = ds[ep_from[e]]
    r = analyse(s); r["ep"] = e; r["task"] = s["prompt"].split("instruction:")[-1].strip()[:70]; cross.append(r)
    print(f"\r{i+1}/{len(eps)} {time.time()-t0:.0f}s", end="")
print()
pick_c = collections.Counter(r["pick"] for r in cross)
conf = collections.Counter((s, r["looks"][s]) for r in cross for s in STREAMS)
picked_looks = collections.Counter((r["pick"], r["looks"][r["pick"]]) for r in cross)
print("router picks:", dict(pick_c))
print("\nwhat each slot looks like (rows = slot, cols = best-matching GT), over all windows:")
print(f"{'slot':8s}" + "".join(f"{g:>8s}" for g in STREAMS))
for s in STREAMS: print(f"{s:8s}" + "".join(f"{conf[(s, g)]:8d}" for g in STREAMS))
print("\npicked slot -> looks like:", dict(picked_looks))
with open(f"{OUT}/cross_tasks.tsv", "w") as f:
    f.write("episode\ttask\tpick\tp_rgb\tp_depth\tp_flow\trgb_looks\tdepth_looks\tflow_looks\n")
    for r in cross:
        f.write(f"{r['ep']}\t{r['task']}\t{r['pick']}\t" + "\t".join(f"{p:.3f}" for p in r["probs"]) + "\t"
                + "\t".join(r["looks"][s] for s in STREAMS) + "\n")
print("saved", f"{OUT}/cross_tasks.tsv")